In [1]:
from app.agent.graph import graph

result = graph.invoke({
    "question": "پرفروش ترین محصول چیست؟"
})

print(result)

{'question': 'پرفروش ترین محصول چیست؟', 'mode': 'full', 'intro_message': 'کوئری برای یافتن پرفروش\u200cترین محصول آماده شده است.', 'sql_message': 'این کوئری نام محصولی را که بیشترین تعداد فروش را داشته است، به همراه مجموع تعداد فروش آن نمایش می\u200cدهد. نتیجه این کوئری در ادامه آمده است.', 'sql': 'SELECT p.product_name, SUM(oi.quantity) AS total_quantity_sold\nFROM sales.order_items oi\nJOIN production.products p ON oi.product_id = p.product_id\nGROUP BY p.product_name\nORDER BY total_quantity_sold DESC\nLIMIT 1;', 'result': [{'product_name': 'نوکیا 3310', 'total_quantity_sold': 30}], 'analysis': 'محصول پرفروش در این مجموعه داده، "نوکیا 3310" است.\n\nاین محصول با مجموع فروش 30 واحد، بیشترین تعداد فروش را در بین محصولات دیگر داشته است.\n\nنتیجه نشان می\u200cدهد که "نوکیا 3310" در صدر لیست محصولات پرفروش قرار دارد.', 'error': None}


In [1]:
from app.agent.graph import graph

inputs = {
    "question": "پرفروش ترین محصول چیست؟"
}

for event in graph.stream(inputs):
    print("="*50)
    print(event)

{'intent': {'mode': 'full'}}
{'full': {'intro_message': 'کوئری زیر برای یافتن پرفروش\u200cترین محصول آماده شده است.', 'sql': 'SELECT p.product_name, SUM(oi.quantity) AS total_quantity_sold\nFROM production.products p\nJOIN sales.order_items oi ON p.product_id = oi.product_id\nGROUP BY p.product_name\nORDER BY total_quantity_sold DESC\nLIMIT 1;', 'sql_message': 'این کوئری نام محصولی را که بیشترین تعداد فروش را داشته است، به همراه مجموع تعداد فروش آن نمایش می\u200cدهد. نتیجه این کوئری در ادامه آمده است.'}}
{'execute_sql': {'result': [{'product_name': 'Cannondale Topstone', 'total_quantity_sold': 2}], 'error': None}}
{'analyzer': {'analysis': 'محصول "Cannondale Topstone" به عنوان پرفروش\u200cترین محصول شناخته شده است.\n\nتعداد کل فروش این محصول برابر با ۲ واحد می\u200cباشد.\n\nاین نتیجه نشان می\u200cدهد که در میان محصولات موجود، "Cannondale Topstone" بیشترین تعداد فروش را داشته است.'}}


In [2]:
from app.agent.graph import graph

inputs = {
    "question": "سلام"
}

for event in graph.stream(inputs):
    print("="*50)
    print(event)

{'intent': {'mode': 'chat'}}
{'chat': {'message': 'سلام! من اینجا هستم تا به شما در مورد سوالات دیتابیس و SQL کمک کنم. چطور می\u200cتونم کمکتون کنم؟'}}


In [3]:
from app.agent.graph import graph

inputs = {
    "question": "فقط کوعری 3 مشتری برتر رو بده"
}

for event in graph.stream(inputs):
    print("="*50)
    print(event)

{'intent': {'mode': 'sql'}}
{'sql': {'sql': 'SELECT c.customer_id, c.first_name, c.last_name, SUM(oi.list_price * oi.quantity * (1 - oi.discount)) AS total_spent\nFROM sales.customers c\nJOIN sales.orders o ON c.customer_id = o.customer_id\nJOIN sales.order_items oi ON o.order_id = oi.order_id\nGROUP BY c.customer_id, c.first_name, c.last_name\nORDER BY total_spent DESC\nLIMIT 3;'}}


In [4]:
from app.agent.graph import graph

inputs = {
    "question": "فقط نتیجه سه مشتری برتر رو بده"
}

for event in graph.stream(inputs):
    print("="*50)
    print(event)

{'intent': {'mode': 'result'}}
{'sql': {'sql': 'SELECT c.customer_id, c.first_name, c.last_name, SUM(oi.list_price * oi.quantity * (1 - oi.discount)) AS total_spent\nFROM sales.customers c\nJOIN sales.orders o ON c.customer_id = o.customer_id\nJOIN sales.order_items oi ON o.order_id = oi.order_id\nGROUP BY c.customer_id, c.first_name, c.last_name\nORDER BY total_spent DESC\nLIMIT 3;'}}
{'execute_sql': {'result': [{'customer_id': 4, 'first_name': 'Maryam', 'last_name': 'Rad', 'total_spent': Decimal('5500.0000')}, {'customer_id': 2, 'first_name': 'Sara', 'last_name': 'Ahmadi', 'total_spent': Decimal('4200.0000')}, {'customer_id': 5, 'first_name': 'Omid', 'last_name': 'Sadeghi', 'total_spent': Decimal('3906.0000')}], 'error': None}}
{'analyzer': {'analysis': 'مشتری اول با بیشترین میزان خرید، مریم راد با شناسه مشتری ۴ است که مجموع خریدهای او به مبلغ ۵۵۰۰ واحد می\u200cرسد.\n\nمشتری دوم سارا احمدی با شناسه مشتری ۲ است که مجموع خریدهای او ۴۲۰۰ واحد است.\n\nمشتری سوم امید صادقی با شناسه مشتری 

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from app.core.config import settings
from app.agent.schemas  import (
    IntentOutput,
    ChatOutput
)


#LLM Setup 

llm = ChatOpenAI(
    base_url="https://api.gapgpt.app/v1",
    api_key=settings.OPENAI_API_KEY,
    model="gpt-4o",
    temperature=0,
    streaming=False
)

streaming_llm = ChatOpenAI(
    base_url="https://api.gapgpt.app/v1",
    api_key=settings.OPENAI_API_KEY,
    model="gpt-4o",
    temperature=0,
    streaming=True
)


#Prompts 

intent_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an intent classifier for a SQL agent.

Classify the user request into one of these modes:

chat   → greeting or unrelated to database
sql    → user explicitly asks for SQL query only
result → user wants only the raw result
full   → default for any data question

Important:
If the user asks about data, ranking, statistics, counts, etc.,
and does NOT explicitly request SQL only,
you MUST return full.
"""),
    ("human", "{question}")
])

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a SQL intelligent assistant connected to the user's database.
You MUST always respond in Persian (Farsi) language only. Never respond in English or any other language.

Mirror the user's tone and formality:
- If they're casual and warm, be friendly and conversational.
- If they're direct and brief, keep it short and to the point.
- If they greet you, greet back warmly, then briefly mention what you do.
- If they don't greet you, never greet them.
- If they jump straight to questions, skip the pleasantries and get to work.

Your role:
You help users query their SQL database through natural language. Users can ask questions and receive:
- SQL queries
- Query results in table format
- Analysis and insights

Stay focused:
Only answer SQL and database-related questions.
If asked something irrelevant, politely redirect in Persian: "من فقط می‌توانم در مورد سوالات دیتابیس و SQL کمک کنم."

"""),
    ("human", "{question}")
])


intent_chain = intent_prompt | llm.with_structured_output(IntentOutput)
chat_chain = chat_prompt | llm.with_structured_output(ChatOutput)


In [6]:
from typing import Dict, List, Any
from app.agent.schemas.states  import AgentState
from app.agent.chains import (
    intent_chain,
    chat_chain,
    sql_chain,
    full_chain,
    analyzer_chain,
)
from app.core.database.clientdb import get_db_schema_text , run_sql_query


#Nodes

def intent_node(state: AgentState):
    result = intent_chain.invoke({"question": state["question"]})
    return {"mode": result.mode}


def router(state: AgentState):
    return state["mode"]


def chat_node(state: AgentState):
    result = chat_chain.invoke({"question": state["question"]})
    return {"message": result.message}




In [7]:
from langgraph.graph import StateGraph, END

from app.agent.schemas.states  import AgentState
from app.agent.nodes import (
    intent_node,
    router,
    chat_node,
    sql_node,
    full_node,
    execute_sql_node,
    after_sql_router,
    analyzer_node,
)


#Graph

def build_graph():
    builder = StateGraph(AgentState)

    builder.add_node("intent", intent_node)
    builder.add_node("chat", chat_node)


    builder.set_entry_point("intent")

    builder.add_conditional_edges(
        "intent",
        router,
        {
            "chat": "chat"
        }
    )
    return builder.compile()


graph = build_graph()

In [9]:
inputs = {
    "question": "سلام"
}
graph.invoke(inputs)

{'question': 'سلام',
 'mode': 'chat',
 'message': 'سلام! من می\u200cتونم به شما در نوشتن کوئری\u200cهای SQL و تحلیل داده\u200cها کمک کنم. چطور می\u200cتونم کمکتون کنم؟'}

In [10]:
inputs = {
    "question": "سلام"
}

for event in graph.stream(inputs):
    print("="*50)
    print(event)

{'intent': {'mode': 'chat'}}
{'chat': {'message': 'سلام! من اینجا هستم تا به شما در نوشتن و اجرای کوئری\u200cهای SQL کمک کنم. چطور می\u200cتونم کمکتون کنم؟'}}


In [2]:
import asyncio
from typing import Optional, Any
from typing_extensions import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, END
from langgraph.types import StreamWriter

# ─────────────────────────────────────────
# Config
# ─────────────────────────────────────────
BASE_URL = "https://api.gapgpt.app/v1"
API_KEY  = "sk-s8KnoW59PPxeHBvyzENeVoEiH2QbiNm1PxJt20H586up5p8n"
MODEL    = "gpt-4o"

# ─────────────────────────────────────────
# LLMs
# ─────────────────────────────────────────
llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
    temperature=0,
    streaming=False,
)

streaming_llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
    temperature=0,
    streaming=True,
)

# ─────────────────────────────────────────
# Schemas
# ─────────────────────────────────────────
class IntentOutput(BaseModel):
    mode: str = Field(description="chat | sql | result | full")

# ─────────────────────────────────────────
# State
# ─────────────────────────────────────────
class TestState(TypedDict):
    question: str
    mode:     Optional[str]
    message:  Optional[str]

# ─────────────────────────────────────────
# Prompts & Chains
# ─────────────────────────────────────────
intent_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an intent classifier for a SQL agent.
Classify the user request into one of these modes:
chat   → greeting or unrelated to database
sql    → user explicitly asks for SQL query only
result → user wants only the raw result
full   → default for any data question
"""),
    ("human", "{question}"),
])

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a SQL intelligent assistant.
You MUST always respond in Persian (Farsi) language only.
Only answer SQL and database-related questions.
"""),
    ("human", "{question}"),
])

intent_chain = intent_prompt | llm.with_structured_output(IntentOutput)
chat_chain   = chat_prompt   | streaming_llm   # بدون structured_output

# ─────────────────────────────────────────
# Nodes
# ─────────────────────────────────────────
def test_intent_node(state: TestState):
    result = intent_chain.invoke({"question": state["question"]})
    return {"mode": result.mode}

def test_router(state: TestState):
    return state["mode"]

def test_chat_node(state: TestState, writer: StreamWriter):
    full_message = ""

    for chunk in chat_chain.stream({"question": state["question"]}):
        token = chunk.content
        if token:
            full_message += token
            writer({"type": "token", "value": token})

    return {"message": full_message}

# ─────────────────────────────────────────
# Graph
# ─────────────────────────────────────────
def build_test_graph():
    builder = StateGraph(TestState)

    builder.add_node("test_intent", test_intent_node)
    builder.add_node("test_chat",   test_chat_node)

    builder.set_entry_point("test_intent")

    builder.add_conditional_edges(
        "test_intent",
        test_router,
        {"chat": "test_chat"},
    )

    builder.add_edge("test_chat", END)

    return builder.compile()

test_graph = build_test_graph()

# ─────────────────────────────────────────
# Run
# ─────────────────────────────────────────
inputs = {"question": "سلام"}

for event in test_graph.stream(inputs, stream_mode=["updates", "custom"]):
    mode, data = event

    if mode == "updates":
        print(data)

    elif mode == "custom":
        print({"chat": {"custom": data}})

{'test_intent': {'mode': 'chat'}}
{'chat': {'custom': {'type': 'token', 'value': 'سلام'}}}
{'chat': {'custom': {'type': 'token', 'value': '!'}}}
{'chat': {'custom': {'type': 'token', 'value': ' چ'}}}
{'chat': {'custom': {'type': 'token', 'value': 'طور'}}}
{'chat': {'custom': {'type': 'token', 'value': ' می'}}}
{'chat': {'custom': {'type': 'token', 'value': '\u200cتوان'}}}
{'chat': {'custom': {'type': 'token', 'value': 'م'}}}
{'chat': {'custom': {'type': 'token', 'value': ' در'}}}
{'chat': {'custom': {'type': 'token', 'value': ' زمینه'}}}
{'chat': {'custom': {'type': 'token', 'value': ' SQL'}}}
{'chat': {'custom': {'type': 'token', 'value': ' یا'}}}
{'chat': {'custom': {'type': 'token', 'value': ' پای'}}}
{'chat': {'custom': {'type': 'token', 'value': 'گاه'}}}
{'chat': {'custom': {'type': 'token', 'value': '\u200c'}}}
{'chat': {'custom': {'type': 'token', 'value': 'د'}}}
{'chat': {'custom': {'type': 'token', 'value': 'اده'}}}
{'chat': {'custom': {'type': 'token', 'value': ' به'}}}
{'cha